<style>
:root {
  --ink: #17202a;
  --muted: #64748b;
  --paper: #fbf7ef;
  --card: #fffaf0;
  --line: #e7dcc9;
  --navy: #11263f;
  --teal: #0f766e;
  --amber: #b45309;
  --rose: #be123c;
}
body, .jp-Notebook { background: var(--paper); color: var(--ink); }
.jp-RenderedHTMLCommon h1 { color: var(--navy); font-size: 2.2rem; letter-spacing: -0.03em; }
.jp-RenderedHTMLCommon h2 { color: var(--teal); margin-top: 2rem; }
.jp-RenderedHTMLCommon h3 { color: var(--navy); }
.jp-RenderedHTMLCommon p, .jp-RenderedHTMLCommon li { font-size: 1rem; line-height: 1.55; }
.jp-RenderedHTMLCommon code { background: #f1e7d6; color: #7c2d12; padding: 0.12rem 0.28rem; border-radius: 0.25rem; }
.jp-RenderedHTMLCommon table { border-collapse: collapse; width: 100%; background: var(--card); }
.jp-RenderedHTMLCommon th { background: #efe3cd; color: var(--navy); }
.jp-RenderedHTMLCommon th, .jp-RenderedHTMLCommon td { border: 1px solid var(--line); padding: 0.45rem 0.55rem; }
.note-box { background: #fff7ed; border: 1px solid #fed7aa; border-left: 6px solid var(--amber); padding: 0.9rem 1rem; border-radius: 0.65rem; }
.result-box { background: #ecfdf5; border: 1px solid #99f6e4; border-left: 6px solid var(--teal); padding: 0.9rem 1rem; border-radius: 0.65rem; }
</style>


# Validación temporal de Rossmann con Jano

<div class="note-box">
Este notebook es el primer candidato a ejemplo de oro para Jano. Usa el dataset Rossmann Store Sales para comparar una partición aleatoria, un holdout cronológico estático y una simulación walk-forward auditable.
</div>

El objetivo no es ganar la competencia de Kaggle. El objetivo es mostrar cómo Jano ayuda a responder una pregunta más operativa:

> ¿Este modelo se habría comportado de forma consistente en el tiempo bajo una política específica de reentrenamiento y evaluación?

Fuente del dataset: [Rossmann Store Sales en Kaggle](https://www.kaggle.com/c/rossmann-store-sales). Este notebook también soporta el mirror público de Kaggle [`pratyushakar/rossmann-store-sales`](https://www.kaggle.com/datasets/pratyushakar/rossmann-store-sales), que incluye `train.csv`, `test.csv` y `store.csv`.


## 0. Preparación

Los archivos pesados del dataset viven en `data/raw/rossmann/`, que está ignorado por git. Si no tenés los archivos locales, configurá primero la Kaggle CLI:

```bash
pip install kaggle
mkdir -p ~/.kaggle
# colocá kaggle.json en ~/.kaggle/ y ejecutá chmod 600 ~/.kaggle/kaggle.json
```

Después ejecutá la celda de descarga de abajo. También podés usar el helper de datasets de Jano:

```bash
python scripts/download_dataset.py rossmann_store_sales --extract
```

Si ya tenés `train.csv` y `store.csv`, colocalos en `data/raw/rossmann/` y salteate la descarga.


In [1]:
from __future__ import annotations

from pathlib import Path
import subprocess
import sys
import zipfile

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "jano").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from jano import (
    EvaluationProfile,
    PeriodicRetrain,
    TemporalPartitionSpec,
    WalkForwardPolicy,
    WalkForwardRunner,
)

DATA_DIR = PROJECT_ROOT / "data" / "raw" / "rossmann"
TRAIN_CSV = DATA_DIR / "train.csv"
STORE_CSV = DATA_DIR / "store.csv"
RANDOM_SEED = 42


In [2]:
def ensure_rossmann_files() -> bool:
    """Return True when real Rossmann files are available locally."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    if TRAIN_CSV.exists() and STORE_CSV.exists():
        print(f"Los archivos de Rossmann ya existen en {DATA_DIR}")
        return True

    command = [
        "kaggle",
        "datasets",
        "download",
        "-d",
        "pratyushakar/rossmann-store-sales",
        "-p",
        str(DATA_DIR),
    ]
    try:
        subprocess.run(command, check=True)
    except (FileNotFoundError, subprocess.CalledProcessError):
        print("Los archivos de Kaggle no están disponibles en este entorno.")
        print("El notebook usará un dataset determinista similar a Rossmann para que Jano pueda ejecutarse de punta a punta.")
        return False

    for archive in DATA_DIR.glob("*.zip"):
        with zipfile.ZipFile(archive) as zf:
            zf.extractall(DATA_DIR)

    available = TRAIN_CSV.exists() and STORE_CSV.exists()
    if not available:
        print("El archivo descargado no expuso train.csv y store.csv; se usará el fallback sintético.")
    return available


HAS_REAL_ROSSMANN = ensure_rossmann_files()
DATASET_SOURCE = "Rossmann Store Sales de Kaggle" if HAS_REAL_ROSSMANN else "Fallback sintético tipo Rossmann"
print("Fuente del dataset:", DATASET_SOURCE)


Los archivos de Kaggle no están disponibles en este entorno.
El notebook usará un dataset determinista similar a Rossmann para que Jano pueda ejecutarse de punta a punta.
Fuente del dataset: Fallback sintético tipo Rossmann


## 1. Cargar y preparar un frame parecido a producción

Rossmann tiene una fila por tienda y por día. El target es `Sales`. Evitamos deliberadamente `Customers` porque no está disponible al momento de predecir en el test set de Kaggle, así que usarlo generaría un problema de disponibilidad de features.

Para mantener el notebook rápido, usamos un subconjunto configurable de tiendas. La estructura temporal se preserva: cada tienda seleccionada mantiene su historial completo de fechas.


In [3]:
def build_synthetic_rossmann_frame(
    *,
    stores: int = 180,
    start: str = "2014-01-01",
    end: str = "2015-07-31",
    seed: int = RANDOM_SEED,
) -> pd.DataFrame:
    """Crear un panel diario determinista similar a Rossmann cuando no hay archivos de Kaggle disponibles."""
    rng = np.random.default_rng(seed)
    dates = pd.date_range(start, end, freq="D")
    store_ids = np.arange(1, stores + 1)
    grid = pd.MultiIndex.from_product([dates, store_ids], names=["Date", "Store"]).to_frame(index=False)

    store_attrs = pd.DataFrame(
        {
            "Store": store_ids,
            "StoreType": rng.choice(list("abcd"), size=stores, p=[0.45, 0.18, 0.25, 0.12]),
            "Assortment": rng.choice(list("abc"), size=stores, p=[0.50, 0.15, 0.35]),
            "CompetitionDistance": rng.gamma(shape=2.5, scale=900, size=stores).round(0),
        }
    )
    frame = grid.merge(store_attrs, on="Store", how="left")
    frame["DayOfWeek"] = frame["Date"].dt.dayofweek + 1
    frame["Month"] = frame["Date"].dt.month
    frame["WeekOfYear"] = frame["Date"].dt.isocalendar().week.astype(int)
    frame["Promo"] = ((frame["WeekOfYear"] % 4).isin([1, 2]) & (frame["DayOfWeek"] <= 5)).astype(int)
    frame["SchoolHoliday"] = frame["Month"].isin([7, 8, 12]).astype(int)
    frame["StateHoliday"] = np.where(frame["Date"].dt.strftime("%m-%d").isin(["01-01", "12-25"]), "a", "none")
    frame["Open"] = (frame["DayOfWeek"] != 7).astype(int)

    base_by_store = rng.normal(6200, 950, size=stores)
    store_effect = pd.Series(base_by_store, index=store_ids)
    dow_effect = frame["DayOfWeek"].map({1: 350, 2: 250, 3: 150, 4: 100, 5: 450, 6: -700, 7: -2200}).to_numpy()
    promo_effect = frame["Promo"].to_numpy() * rng.normal(1050, 140, size=len(frame))
    seasonal = 550 * np.sin(2 * np.pi * frame["Date"].dt.dayofyear.to_numpy() / 365.25)
    drift = np.where(frame["Date"] >= pd.Timestamp("2015-01-01"), 350, 0)
    noise = rng.normal(0, 420, size=len(frame))
    sales = (
        frame["Store"].map(store_effect).to_numpy()
        + dow_effect
        + promo_effect
        + seasonal
        + drift
        - 0.04 * frame["CompetitionDistance"].to_numpy()
        + noise
    )
    frame["Sales"] = np.where(frame["Open"] == 1, np.maximum(sales, 0), 0).round(0).astype(int)
    return frame.loc[(frame["Open"] == 1) & (frame["Sales"] > 0)].sort_values(["Date", "Store"]).reset_index(drop=True)


def load_rossmann_frame(store_limit: int = 180) -> pd.DataFrame:
    if not HAS_REAL_ROSSMANN:
        return build_synthetic_rossmann_frame(stores=store_limit)

    sales = pd.read_csv(TRAIN_CSV, parse_dates=["Date"], low_memory=False)
    stores = pd.read_csv(STORE_CSV)

    selected_stores = (
        sales.groupby("Store")["Date"]
        .nunique()
        .sort_values(ascending=False)
        .head(store_limit)
        .index
    )
    frame = sales.loc[sales["Store"].isin(selected_stores)].merge(stores, on="Store", how="left")
    frame = frame.loc[(frame["Open"].fillna(1) == 1) & (frame["Sales"] > 0)].copy()
    frame = frame.sort_values(["Date", "Store"]).reset_index(drop=True)

    frame["StateHoliday"] = frame["StateHoliday"].astype(str).replace({"0": "none", "0.0": "none"})
    frame["StoreType"] = frame["StoreType"].fillna("unknown")
    frame["Assortment"] = frame["Assortment"].fillna("unknown")
    frame["CompetitionDistance"] = frame["CompetitionDistance"].fillna(frame["CompetitionDistance"].median())
    frame["Month"] = frame["Date"].dt.month
    frame["WeekOfYear"] = frame["Date"].dt.isocalendar().week.astype(int)

    return frame


rossmann = load_rossmann_frame(store_limit=180)
print("Fuente del dataset:", DATASET_SOURCE)
print(f"rows={len(rossmann):,} stores={rossmann['Store'].nunique():,} date_range={rossmann['Date'].min().date()} -> {rossmann['Date'].max().date()}")
display(rossmann.head())


Fuente del dataset: Fallback sintético tipo Rossmann
rows=89,100 stores=180 date_range=2014-01-01 -> 2015-07-31


,Date,Store,StoreType,Assortment,CompetitionDistance,DayOfWeek,Month,WeekOfYear,Promo,SchoolHoliday,StateHoliday,Open,Sales
0,2014-01-01,1,c,c,1267.0,3,1,1,1,0,a,1,6368
1,2014-01-01,2,a,c,516.0,3,1,1,1,0,a,1,6641
2,2014-01-01,3,c,b,2480.0,3,1,1,1,0,a,1,8358
3,2014-01-01,4,c,a,1287.0,3,1,1,1,0,a,1,6962
4,2014-01-01,5,a,c,3736.0,3,1,1,1,0,a,1,6354


## 2. Un modelo simple para que el foco siga en la validación

Este notebook usa un modelo mediano simple implementado aquí. Memoriza la mediana de ventas por tienda, día de la semana, promo y atributos de tienda, y luego cae a medianas más generales cuando un grupo no fue visto.

Eso mantiene el ejemplo liviano en dependencias. Podés reemplazar esta clase por cualquier estimador que exponga `fit(X, y)` y `predict(X)`.


In [4]:
FEATURE_COLS = [
    "Store",
    "DayOfWeek",
    "Promo",
    "SchoolHoliday",
    "StateHoliday",
    "StoreType",
    "Assortment",
    "CompetitionDistance",
    "Month",
]


class GroupMedianRegressor:
    def __init__(self, group_cols: list[str], fallback_cols: list[str] | None = None) -> None:
        self.group_cols = list(group_cols)
        self.fallback_cols = list(fallback_cols or [])

    def fit(self, X: pd.DataFrame, y: pd.Series):
        frame = X.copy()
        frame["__target__"] = np.asarray(y, dtype=float)
        self.global_median_ = float(frame["__target__"].median())
        self.group_medians_ = frame.groupby(self.group_cols, dropna=False)["__target__"].median()
        if self.fallback_cols:
            self.fallback_medians_ = frame.groupby(self.fallback_cols, dropna=False)["__target__"].median()
        else:
            self.fallback_medians_ = None
        return self

    def _lookup(self, X: pd.DataFrame, cols: list[str], medians: pd.Series) -> np.ndarray:
        keys = pd.MultiIndex.from_frame(X[cols])
        return medians.reindex(keys).to_numpy(dtype=float)

    def predict(self, X: pd.DataFrame) -> np.ndarray:
        predictions = self._lookup(X, self.group_cols, self.group_medians_)
        missing = np.isnan(predictions)
        if missing.any() and self.fallback_medians_ is not None:
            fallback = self._lookup(X.loc[missing], self.fallback_cols, self.fallback_medians_)
            predictions[missing] = fallback
            missing = np.isnan(predictions)
        predictions[missing] = self.global_median_
        return predictions


def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.asarray(y_true) - np.asarray(y_pred))))


def rmspe(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    denominator = np.maximum(y_true, 1.0)
    return float(np.sqrt(np.mean(((y_true - y_pred) / denominator) ** 2)))


model = GroupMedianRegressor(
    group_cols=["Store", "DayOfWeek", "Promo", "SchoolHoliday", "StateHoliday"],
    fallback_cols=["Store", "DayOfWeek", "Promo"],
)


## 3. Línea base: partición aleatoria

Una partición aleatoria no es automáticamente incorrecta. Lo es cuando la pregunta de fondo es temporal: entrenar con el pasado y evaluar sobre un horizonte futuro. Acá inspeccionamos qué hace realmente la partición con las fechas.


In [5]:
def evaluate_indices(name: str, frame: pd.DataFrame, train_idx: np.ndarray, test_idx: np.ndarray) -> dict[str, object]:
    estimator = GroupMedianRegressor(
        group_cols=["Store", "DayOfWeek", "Promo", "SchoolHoliday", "StateHoliday"],
        fallback_cols=["Store", "DayOfWeek", "Promo"],
    )
    train = frame.iloc[train_idx]
    test = frame.iloc[test_idx]
    estimator.fit(train[FEATURE_COLS], train["Sales"])
    prediction = estimator.predict(test[FEATURE_COLS])
    return {
        "protocol": name,
        "train_rows": len(train),
        "test_rows": len(test),
        "train_start": train["Date"].min(),
        "train_end": train["Date"].max(),
        "test_start": test["Date"].min(),
        "test_end": test["Date"].max(),
        "mae": mae(test["Sales"], prediction),
        "rmspe": rmspe(test["Sales"], prediction),
    }


rng = np.random.default_rng(RANDOM_SEED)
indices = rng.permutation(len(rossmann))
cut = int(len(indices) * 0.8)
random_result = evaluate_indices("random_80_20", rossmann, indices[:cut], indices[cut:])
display(
    pd.DataFrame([random_result]).rename(
        columns={
            "protocol": "protocolo",
            "train_rows": "filas_entrenamiento",
            "test_rows": "filas_prueba",
            "train_start": "inicio_entrenamiento",
            "train_end": "fin_entrenamiento",
            "test_start": "inicio_prueba",
            "test_end": "fin_prueba",
        }
    )
)


,protocolo,filas_entrenamiento,filas_prueba,inicio_entrenamiento,fin_entrenamiento,inicio_prueba,fin_prueba,mae,rmspe
0,random_80_20,71280,17820,2014-01-01,2015-07-31,2014-01-01,2015-07-31,498.16748,0.102699


## 4. Línea base: holdout cronológico estático

Un holdout cronológico es una mejor línea base operativa: entrena con el pasado y prueba sobre las últimas seis semanas. Igual sigue dando una sola foto. No muestra si el rendimiento fue estable en varias fechas de despliegue.


In [6]:
holdout_days = 42
holdout_start = rossmann["Date"].max() - pd.Timedelta(days=holdout_days - 1)
holdout_train_idx = rossmann.index[rossmann["Date"] < holdout_start].to_numpy()
holdout_test_idx = rossmann.index[rossmann["Date"] >= holdout_start].to_numpy()

baseline_comparison = pd.DataFrame([
    random_result,
    evaluate_indices("last_42_days_holdout", rossmann, holdout_train_idx, holdout_test_idx),
])
display(
    baseline_comparison.rename(
        columns={
            "protocol": "protocolo",
            "train_rows": "filas_entrenamiento",
            "test_rows": "filas_prueba",
            "train_start": "inicio_entrenamiento",
            "train_end": "fin_entrenamiento",
            "test_start": "inicio_prueba",
            "test_end": "fin_prueba",
        }
    )
)

random_temporal_overlap = (
    baseline_comparison.loc[0, "train_end"] >= baseline_comparison.loc[0, "test_start"]
    and baseline_comparison.loc[0, "test_end"] >= baseline_comparison.loc[0, "train_start"]
)
print("Interpretación:")
print(f"- Las ventanas temporales de train y test de la partición aleatoria se superponen: {random_temporal_overlap}")
print("- El holdout cronológico responde una pregunta más parecida a producción, pero solo para una ventana futura.")


,protocolo,filas_entrenamiento,filas_prueba,inicio_entrenamiento,fin_entrenamiento,inicio_prueba,fin_prueba,mae,rmspe
0,random_80_20,71280,17820,2014-01-01,2015-07-31,2014-01-01,2015-07-31,498.167480,0.102699
1,last_42_days_holdout,82620,6480,2014-01-01,2015-06-19,2015-06-20,2015-07-31,486.530015,0.087664


Interpretación:
- Las ventanas temporales de train y test de la partición aleatoria se superponen: True
- El holdout cronológico responde una pregunta más parecida a producción, pero solo para una ventana futura.


## 4. TimeSeriesSplit: la primitiva básica de walk-forward

`TimeSeriesSplit` es una línea base walk-forward válida. Conserva el orden temporal y te permite hacer crecer la ventana de entrenamiento mientras movés la ventana de prueba hacia adelante. Eso sirve para una validación secuencial rápida.

Lo que **no** te da es el contrato temporal más amplio que Jano modela como un objeto de primera clase: un plan que podés inspeccionar, filtrar, acotar con anclas de inicio y fin, componer con reglas de reentrenamiento y luego materializar en una simulación.

La idea acá no es que `TimeSeriesSplit` esté mal. La idea es que resuelve un problema más chico.


In [7]:
try:
    from sklearn.model_selection import TimeSeriesSplit
except ModuleNotFoundError:
    TimeSeriesSplit = None

# Una línea base secuencial sobre el mismo caso Rossmann.
# Para que la geometría temporal se vea clara, usamos un eje de fechas único.

def iter_time_series_splits(n_samples, n_splits, test_size, gap=0, max_train_size=None):
    if n_splits < 1:
        raise ValueError("n_splits must be at least 1")
    step = test_size
    test_starts = [n_samples - n_splits * test_size + i * step for i in range(n_splits)]
    for test_start in test_starts:
        test_end = test_start + test_size
        train_end = test_start - gap
        train_start = 0 if max_train_size is None else max(0, train_end - max_train_size)
        yield np.arange(train_start, train_end), np.arange(test_start, test_end)

time_axis = rossmann.sort_values("Date").drop_duplicates("Date")[ ["Date"] ].reset_index(drop=True)

if TimeSeriesSplit is not None:
    splitter = TimeSeriesSplit(n_splits=4, test_size=14, gap=1, max_train_size=60)
    split_iter = splitter.split(time_axis)
else:
    split_iter = iter_time_series_splits(len(time_axis), n_splits=4, test_size=14, gap=1, max_train_size=60)

rows = []
for fold, (train_idx, test_idx) in enumerate(split_iter):
    train = time_axis.iloc[train_idx]
    test = time_axis.iloc[test_idx]
    rows.append(
        {
            "pliegue": fold,
            "inicio_entrenamiento": train["Date"].min().date().isoformat(),
            "fin_entrenamiento": train["Date"].max().date().isoformat(),
            "filas_entrenamiento": len(train),
            "inicio_prueba": test["Date"].min().date().isoformat(),
            "fin_prueba": test["Date"].max().date().isoformat(),
            "filas_prueba": len(test),
        }
    )

tscv_summary = pd.DataFrame(rows)
print(tscv_summary.to_string(index=False))


 pliegue inicio_entrenamiento fin_entrenamiento  filas_entrenamiento inicio_prueba fin_prueba  filas_prueba
       0           2015-03-18        2015-05-26                   60    2015-05-28 2015-06-12            14
       1           2015-04-03        2015-06-11                   60    2015-06-13 2015-06-29            14
       2           2015-04-20        2015-06-27                   60    2015-06-30 2015-07-15            14
       3           2015-05-06        2015-07-14                   60    2015-07-16 2015-07-31            14


## 5. Planificar la simulación walk-forward antes de entrenar

Acá entra Jano. [`plan()`](https://marmurar.github.io/jano/concepts.html#planning-before-materialization) precalcula la geometría de los folds antes de materializar slices de train/test o de ajustar un modelo.

Acá la policy significa:

- usar 180 días de historial de entrenamiento;
- evaluar los siguientes 42 días;
- avanzar 42 días;
- alinear las ventanas a días calendario;
- mantener como máximo ocho folds para un notebook compacto.


In [8]:
policy = WalkForwardPolicy(
    time_col="Date",
    partition=TemporalPartitionSpec(
        layout="train_test",
        train_size="180D",
        test_size="42D",
        calendar_frequency="D",
    ),
    step="42D",
    strategy="rolling",
    start_at="2014-01-01",
    max_folds=8,
)

plan = policy.plan(rossmann, title="Plan walk-forward de Rossmann con 180D de train y 42D de test")
plan_df = plan.to_frame()
display(
    plan_df[["iteration", "fold", "train_start", "train_end", "train_rows", "test_start", "test_end", "test_rows"]].rename(
        columns={
            "iteration": "iteración",
            "fold": "pliegue",
            "train_start": "inicio_entrenamiento",
            "train_end": "fin_entrenamiento",
            "train_rows": "filas_entrenamiento",
            "test_start": "inicio_prueba",
            "test_end": "fin_prueba",
            "test_rows": "filas_prueba",
        }
    )
)


,iteración,pliegue,inicio_entrenamiento,fin_entrenamiento,filas_entrenamiento,inicio_prueba,fin_prueba,filas_prueba
0,0,0,2014-01-01,2014-06-30,27720,2014-06-30,2014-08-11,6480
1,1,1,2014-02-12,2014-08-11,27720,2014-08-11,2014-09-22,6480
2,2,2,2014-03-26,2014-09-22,27720,2014-09-22,2014-11-03,6480
3,3,3,2014-05-07,2014-11-03,27720,2014-11-03,2014-12-15,6480
4,4,4,2014-06-18,2014-12-15,27720,2014-12-15,2015-01-26,6480
5,5,5,2014-07-30,2015-01-26,27720,2015-01-26,2015-03-09,6480
6,6,6,2014-09-10,2015-03-09,27720,2015-03-09,2015-04-20,6480
7,7,7,2014-10-22,2015-04-20,27720,2015-04-20,2015-06-01,6480


## 6. Ejecutar el mismo modelo sobre los folds planificados

[`WalkForwardRunner`](https://marmurar.github.io/jano/api.html#jano.runner.WalkForwardRunner) controla el bucle de fit/predict/métrica. El splitter sigue siendo dueño de la geometría temporal.

Esto mantiene el experimento explícito: mismo modelo, mismas features, misma métrica, múltiples fechas de despliegue temporal.


In [9]:
evaluation = EvaluationProfile(
    metrics={"mae": mae, "rmspe": rmspe},
    metric_directions={"mae": "min", "rmspe": "min"},
    primary_metric="rmspe",
)

runner = WalkForwardRunner(
    model=model,
    target_col="Sales",
    feature_cols=FEATURE_COLS,
    retrain="always",
    evaluation=evaluation,
)

run = runner.run(policy, rossmann)
fold_metrics = run.to_frame()
display(fold_metrics)
display(pd.DataFrame([run.summary()]))

print("Interpretación:")
print("- Cada fila es una fecha de despliegue simulada.")
print("- La trayectoria de métricas muestra si la misma estrategia del modelo se comporta de forma consistente en el tiempo.")
print("- Esta es información que una sola partición aleatoria o un solo holdout no puede mostrar.")


,fold,retrained,last_retrain_fold,train_rows,test_rows,train_start,train_end,test_start,test_end,mae,rmspe
0,0,True,0,27720,6480,2014-01-01,2014-06-30,2014-06-30,2014-08-11,595.588735,0.130740
1,1,True,1,27720,6480,2014-02-12,2014-08-11,2014-08-11,2014-09-22,691.332022,0.151054
2,2,True,2,27720,6480,2014-03-26,2014-09-22,2014-09-22,2014-11-03,815.460185,0.178641
3,3,True,3,27720,6480,2014-05-07,2014-11-03,2014-11-03,2014-12-15,406.995679,0.087519
4,4,True,4,27720,6480,2014-06-18,2014-12-15,2014-12-15,2015-01-26,703.390664,0.123267
5,5,True,5,27720,6480,2014-07-30,2015-01-26,2015-01-26,2015-03-09,1115.937346,0.161852
6,6,True,6,27720,6480,2014-09-10,2015-03-09,2015-03-09,2015-04-20,946.986265,0.143009
7,7,True,7,27720,6480,2014-10-22,2015-04-20,2015-04-20,2015-06-01,423.468981,0.070225


,folds,retrain_policy,retrain_events,retrain_ratio,metrics,primary_metric,mae_mean,mae_best,mae_best_fold,rmspe_mean,rmspe_best,rmspe_best_fold
0,8,AlwaysRetrain,8,1.0,"[mae, rmspe]",rmspe,712.394985,406.995679,3,0.130788,0.070225,7


Interpretación:
- Cada fila es una fecha de despliegue simulada.
- La trayectoria de métricas muestra si la misma estrategia del modelo se comporta de forma consistente en el tiempo.
- Esta es información que una sola partición aleatoria o un solo holdout no puede mostrar.


## 7. Comparar políticas de reentrenamiento

La misma geometría de folds puede responder una segunda pregunta operativa: ¿necesitamos reentrenar en cada fold, o alcanza con una cadencia más barata para este modelo y este dataset?


In [10]:
policy_runs = {}
for label, kwargs in {
    "always": {"retrain": "always"},
    "never": {"retrain": "never"},
    "periodic_2": {"retrain_policy": PeriodicRetrain(2)},
}.items():
    runner = WalkForwardRunner(
        model=model,
        target_col="Sales",
        feature_cols=FEATURE_COLS,
        evaluation=evaluation,
        **kwargs,
    )
    policy_runs[label] = runner.run(policy, rossmann)

comparison = pd.DataFrame(
    {label: result.summary() for label, result in policy_runs.items()}
).T.reset_index(names="policy")
display(
    comparison[["policy", "folds", "retrain_events", "rmspe_mean", "rmspe_best", "rmspe_best_fold", "mae_mean"]].replace(
        {"always": "siempre", "never": "nunca", "periodic_2": "periódico_2"}
    )
)

best_policy = comparison.sort_values("rmspe_mean").iloc[0]
print("Interpretación:")
print(f"- El mejor RMSPE medio en esta corrida fue: {best_policy['policy']} ({best_policy['rmspe_mean']:.4f}).")
print("- El punto importante no es que una política gane siempre; Jano hace explícita y reproducible la comparación de políticas.")


/var/folders/0s/p5lt9bzd49q0f0z226dpx1fc0000gn/T/ipykernel_18493/2973629911.py:20: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  comparison[["policy", "folds", "retrain_events", "rmspe_mean", "rmspe_best", "rmspe_best_fold", "mae_mean"]].replace(


,policy,folds,retrain_events,rmspe_mean,rmspe_best,rmspe_best_fold,mae_mean
0,siempre,8,8,0.130788,0.070225,7,712.394985
1,nunca,8,1,0.123012,0.079614,5,630.880469
2,periódico_2,8,4,0.145853,0.122938,7,807.117564


Interpretación:
- El mejor RMSPE medio en esta corrida fue: never (0.1230).
- El punto importante no es que una política gane siempre; Jano hace explícita y reproducible la comparación de políticas.


In [11]:
trajectory = pd.concat(
    [result.metric_trajectory().assign(policy=label) for label, result in policy_runs.items()],
    ignore_index=True,
)
rmspe_trajectory = trajectory.loc[trajectory["metric"] == "rmspe"]
rmspe_pivot = rmspe_trajectory.pivot(index="fold", columns="policy", values="value")
display(rmspe_pivot)

print("Interpretación:")
print("- Esta tabla ya está lista para graficar: un pliegue por fila, una política de reentrenamiento por columna.")
print("- Herramientas posteriores pueden renderizarla como línea temporal, tarjeta de tablero o reporte sin que Jano imponga el estilo visual.")


policy,always,never,periodic_2
fold,,,
0,0.130740,0.130740,0.130740
1,0.151054,0.172087,0.172087
2,0.178641,0.191745,0.178641
3,0.087519,0.147378,0.127249
4,0.123267,0.089010,0.123267
5,0.161852,0.079614,0.168895
6,0.143009,0.092673,0.143009
7,0.070225,0.080848,0.122938


Interpretación:
- Esta tabla ya está lista para graficar: un pliegue por fila, una política de reentrenamiento por columna.
- Herramientas posteriores pueden renderizarla como línea temporal, tarjeta de tablero o reporte sin que Jano imponga el estilo visual.


## 8. Qué aporta Jano aquí

<div class="result-box">
Jano no afirma que la partición aleatoria sea siempre inválida. Hace testeable la hipótesis temporal. Si el comportamiento de la partición aleatoria, el holdout y el walk-forward es consistente, eso también es evidencia útil. Si divergen, Jano muestra cuándo, dónde y bajo qué política de reentrenamiento.
</div>

En este ejemplo, los artefactos importantes son:

- `plan_df`: la geometría temporal auditable antes de entrenar;
- `fold_metrics`: comportamiento fold por fold bajo una política;
- `comparison`: comparación agregada de políticas de reentrenamiento;
- `trajectory`: datos de métricas listos para graficar para reportes, dashboards o agentes.

Este es el argumento central para paper/landing: la validación temporal debería ser una simulación empírica, no una suposición oculta dentro de una sola partición.
